# StyleGAN — A Style-Based Generator Architecture for GANs (2018)

_Papers · Generative Models_

**Redesign the GAN generator so each layer is steered by a learned 'style', giving scale-separated, controllable image synthesis.**

---

This notebook is a practice scaffold for the **StyleGAN — A Style-Based Generator Architecture for GANs (2018)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, math
torch.manual_seed(0)

# --- 0. Sanity-check the worked AdaIN example (Eq. 1). ---
x = torch.tensor([2., 4., 4., 6.])          # one feature map (4 activations)
ys, yb = 1.5, 0.5                           # style: scale, bias
mu = x.mean(); sigma = x.std(unbiased=False)
xn = (x - mu) / sigma
out = ys * xn + yb
print("worked example:")
print("  mu = %.4f  sigma = %.4f" % (mu, sigma))               # 4.0000  1.4142
print("  normalized:", [round(v,4) for v in xn.tolist()])      # [-1.4142, 0, 0, 1.4142]
print("  styled    :", [round(v,4) for v in out.tolist()])     # [-1.6213, .5, .5, 2.6213]
print("  styled mean = %.4f (== y_b)   std = %.4f (== y_s)" %
      (out.mean(), out.std(unbiased=False)))                   # 0.5000   1.5000


# --- 1. The StyleGAN style pathway, built by hand. ---
Zdim, Wdim, C, HW = 8, 8, 4, 16             # latent dims; channels C; spatial size HW

class Mapping(nn.Module):                   # z -> w  (paper: 8 layers, 512-D; toy: 3 layers)
    def __init__(s):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(Zdim, Wdim), nn.ReLU(),
                              nn.Linear(Wdim, Wdim), nn.ReLU(),
                              nn.Linear(Wdim, Wdim))
    def forward(s, z): return s.net(z)

class Affine(nn.Module):                    # A: w -> (y_s, y_b), one pair per channel
    def __init__(s):
        super().__init__()
        s.lin = nn.Linear(Wdim, 2 * C)
    def forward(s, w):
        y = s.lin(w); return y[:, :C], y[:, C:]      # y_s, y_b   each (B, C)

def adain(x, ys, yb, eps=1e-8):             # x: (B, C, HW)  -- instance norm per (sample, channel)
    mu = x.mean(-1, keepdim=True)
    sd = x.std(-1, keepdim=True, unbiased=False) + eps
    return ys.unsqueeze(-1) * (x - mu) / sd + yb.unsqueeze(-1)

const = nn.Parameter(torch.randn(1, C, HW))         # learned constant input (not z!)
noise_scale = nn.Parameter(torch.zeros(1, C, 1))    # learned per-channel noise weight (B)
mapping, affine = Mapping(), Affine()

def synth(w, add_noise=True):               # one toy synthesis layer: const -> +noise -> AdaIN
    x = const.expand(w.size(0), C, HW).clone()
    if add_noise:
        x = x + noise_scale * torch.randn_like(x)   # per-layer noise injection
    ys, yb = affine(w)
    return adain(x, ys, yb)


# --- 2. DEMO: AdaIN sets each output channel's (mean, std) to the style, content-free. ---
z = torch.randn(1, Zdim); w = mapping(z)
out = synth(w, add_noise=False)
ys, yb = affine(w)
print("\nAdaIN demo: output channel stats track the style")
print("  y_b      :", [round(v,3) for v in yb[0].tolist()])
print("  out mean :", [round(v,3) for v in out[0].mean(-1).tolist()])    # == y_b
print("  |y_s|    :", [round(abs(v),3) for v in ys[0].tolist()])
print("  out std  :", [round(v,3) for v in out[0].std(-1,unbiased=False).tolist()])  # == |y_s|


# --- 3. DEMO: style mixing -- coarse channels from w_A, fine channels from w_B. ---
wA, wB = mapping(torch.randn(1, Zdim)), mapping(torch.randn(1, Zdim))
ysA, ybA = affine(wA); ysB, ybB = affine(wB)
ys_mix = torch.cat([ysA[:, :C//2], ysB[:, C//2:]], 1)   # first half from A, second from B
yb_mix = torch.cat([ybA[:, :C//2], ybB[:, C//2:]], 1)
xconst = const.expand(1, C, HW)
out_mix = adain(xconst, ys_mix, yb_mix)
print("\nStyle mixing: channels 0,1 take A's bias; channels 2,3 take B's bias")
print("  y_b A    :", [round(v,3) for v in ybA[0].tolist()])
print("  y_b B    :", [round(v,3) for v in ybB[0].tolist()])
print("  mix mean :", [round(v,3) for v in out_mix[0].mean(-1).tolist()])


# --- 4. ABLATION: drop the normalize step -> style no longer owns the statistics. ---
def styled_no_norm(x, ys, yb):              # y_s * x + y_b   (NO normalize)
    return ys.unsqueeze(-1) * x + yb.unsqueeze(-1)
content = torch.randn(1, C, HW) * 3.0 + 1.0           # content with its own std ~3
o_adain = adain(content, ys, yb)
o_plain = styled_no_norm(content, ys, yb)
print("\nAblation (channel 0): with AdaIN, out std == |y_s|; without, it rides on content std")
print("  |y_s[0]|        : %.3f" % abs(ys[0,0].item()))
print("  out std AdaIN   : %.3f" % o_adain[0,0].std(unbiased=False).item())   # ~ |y_s|
print("  out std no-norm : %.3f" % o_plain[0,0].std(unbiased=False).item())   # ~ |y_s|*content_std
print("\nAdaIN makes the style OWN each channel's stats; remove the normalize and styles leak.")
# (Exact numbers vary by seed; this is our small demo, not the paper's reported result.)

## Visualize the data & results

_Does AdaIN make a channel's output standard deviation equal the injected style scale y_s — regardless of the content — and what breaks if you drop the normalize step?_

In [ ]:
import torch, torch.nn as nn, numpy as np
torch.manual_seed(0)

# AdaIN (Eq. 1) forces an output channel's std to equal the style scale y_s,
# independent of the content. Drop the normalize -> std rides on the content's std.
C, HW = 1, 64
x = torch.randn(1, C, HW) * 3.7 + 1.2          # arbitrary content (mean 1.2, std 3.7)

def adain(x, ys, yb):
    mu = x.mean(-1, keepdim=True)
    sd = x.std(-1, keepdim=True, unbiased=False)
    return ys * (x - mu) / sd + yb

ys_targets = np.linspace(0.2, 3.0, 20)
yb = 0.5
adain_std  = [round(adain(x, torch.tensor(t), torch.tensor(yb)).std(unbiased=False).item(), 4)
              for t in ys_targets]
adain_mean = [round(adain(x, torch.tensor(t), torch.tensor(yb)).mean().item(), 4)
              for t in ys_targets]
nonorm_std = [round((torch.tensor(t) * x).std(unbiased=False).item(), 4) for t in ys_targets]

print("y_s        :", [round(float(t),3) for t in ys_targets])
print("AdaIN  std :", adain_std)     # == y_s  (lies on y = x)
print("AdaIN  mean:", adain_mean)    # all 0.5 == y_b, content-independent
print("no-norm std:", nonorm_std)    # == y_s * 3.7, off the line
# AdaIN: output std == y_s, mean == y_b, regardless of content. The normalize is what
# makes a style OWN each channel's per-layer statistics -- the key to scale-separated control.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The normalize ablation. In your toy AdaIN, remove the normalize step &mdash; change
            y_s*(x-mu)/sigma + y_b to just y_s*x + y_b. Keep everything else identical.
            Sweep the style scale $\mathbf{y}_s$ from 0.2 to 3.0 on a fixed content channel and measure the
            output std each time. What changes, and what does it show about why AdaIN normalizes first?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- With AdaIN, the normalized map has std 1, so output std $=|\mathbf{y}_s|$ &mdash; it lies exactly on the line "output std = $\mathbf{y}_s$". — _Normalizing erases the content's std, so only the style's scale survives._
- Without the normalize, output std $=|\mathbf{y}_s|\cdot\sigma(\mathbf{x})$ &mdash; the content's own std (here $\approx 3.7$) multiplies in, so the curve is steeper and content-dependent. — _Skipping normalization lets the previous layer's scale leak through; the style no longer owns the statistics._
- Conclude that the normalize step is what makes a style's effect (a) exact and (b) confined to its layer. — _Content-independence + per-layer reset are the whole point of AdaIN; the ablation breaks both._

**Answer:** With AdaIN the output std equals $|\mathbf{y}_s|$ exactly (on the $y=x$ line) and the output mean
                 equals $\mathbf{y}_b$ exactly, regardless of content. Drop the normalize and the output std becomes
                 $|\mathbf{y}_s|\cdot\sigma(\text{content})$ &mdash; off the line and content-dependent &mdash; so styles
                 leak between layers and you lose scale-separated control. The normalize step is what gives the
                 style sole ownership of each channel's per-layer statistics.

</details>

**Problem 2.** Your worked example had $\mathbf{x}_i=[2,4,4,6]$, $\mathbf{y}_s=1.5$, $\mathbf{y}_b=0.5$, giving
            output mean $0.5$ and std $1.5$. Now keep the SAME style but change the content to
            $\mathbf{x}_i=[10,10,10,10]$ shifted/spread differently, say $[7,9,9,11]$ (mean 9, std $\sqrt{2}$).
            What is the AdaIN output mean and std? What does the answer demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Normalize: $\mu=9$, deviations $[-2,0,0,2]$, $\sigma=\sqrt{2}$, so normalized $=[-1.4142,0,0,1.4142]$ &mdash; the SAME normalized map as before. — _Two contents with the same shape but different mean/scale normalize to the same thing._
- Re-style with $\mathbf{y}_s=1.5,\mathbf{y}_b=0.5$: output $=[-1.6213,0.5,0.5,2.6213]$, mean $0.5$, std $1.5$. — _Output statistics are set by the style, not the content._
- Compare to the original example: identical output statistics ($0.5$, $1.5$) despite different input mean (9 vs 4). — _Demonstrates content-independence of the styled statistics._

**Answer:** Output mean $=\mathbf{y}_b=0.5$ and std $=\mathbf{y}_s=1.5$ &mdash; exactly the same as the
                 original example, even though the input mean changed from 4 to 9. This is the key property: after
                 AdaIN, a channel's mean and std are dictated solely by the style $(\mathbf{y}_s,\mathbf{y}_b)$,
                 with the content's original statistics normalized away.

</details>

**Problem 3.** You generate an image with coarse-layer styles from $\mathbf{w}_A$ and fine-layer styles from
            $\mathbf{w}_B$ (style mixing). The result has the pose and face shape of $A$ but the hair/skin
            color scheme of $B$. Explain why, referencing how AdaIN and the layer ordering produce this, and what
            role mixing regularization played in making it clean.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall coarse vs. fine: low-resolution layers (early, small spatial size) set big structure (pose, shape); high-resolution layers (late) set fine texture/color. — _Each layer's style controls that layer's per-channel statistics, and resolution maps to attribute scale._
- Mixing feeds $\mathbf{w}_A$'s styles to the coarse layers and $\mathbf{w}_B$'s to the fine layers, so $A$ owns structure and $B$ owns texture. — _Because AdaIN re-normalizes at each layer, swapping the style source mid-network is clean &mdash; later styles do not depend on earlier ones._
- Note mixing regularization during training (switching $\mathbf{w}$ at a random layer) is what stops the network assuming adjacent layers share a style, making the swap localized. — _&sect;3.1: it "prevents the network from assuming that adjacent styles are correlated", improving localization (Table 2)._

**Answer:** Coarse (low-resolution) layers control structure and fine (high-resolution) layers control
                 texture; mixing routes $\mathbf{w}_A$ to the coarse layers and $\mathbf{w}_B$ to the fine ones,
                 so the image takes $A$'s pose/shape and $B$'s color/texture. The swap is clean because AdaIN
                 normalizes each layer afresh, so a later layer's style does not inherit an earlier one's. Mixing
                 regularization during training &mdash; switching $\mathbf{w}$ at a random layer &mdash; teaches the
                 network not to assume adjacent layers share a style, which is what localizes each style and makes
                 test-time mixing work (&sect;3.1, Table 2).

</details>